In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.image import imread
import os

In [ ]:
# Herramientas de Scikit-learn para métricas de clasificación
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# TensorFlow / Keras: API Sequential, capas convolucionales, modelos prefabricados y utilidades
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization, Resizing, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.saving import load_model

In [ ]:
# Fijar semillas para reproducibilidad de resultados
np.random.seed(42)
tf.random.set_seed(42)

### Análisis Exploratorio y Visualización

In [ ]:
# Ruta al dataset de imágenes de células con malaria
images_path = '/home/alvaro/tensorflow_ejemplos/DATA/cell_images'

In [ ]:
# Construir rutas a los subdirectorios de entrenamiento y prueba
train_path = os.path.join(images_path, 'train')
test_path = os.path.join(images_path, 'test')

In [ ]:
# Listar las clases disponibles en el directorio de entrenamiento
os.listdir(train_path)

In [ ]:
# Visualizar una imagen de ejemplo de la clase 'uninfected' (célula sana)
uninfected_dir = os.path.join(train_path, 'uninfected')
uninfected_cell = os.path.join(uninfected_dir, os.listdir(uninfected_dir)[0])
uninfected_image = imread(uninfected_cell)
plt.imshow(uninfected_image);

In [ ]:
# Visualizar una imagen de ejemplo de la clase 'parasitized' (célula infectada)
parasitized_dir = os.path.join(train_path, 'parasitized')
parasitized_cell = os.path.join(parasitized_dir, os.listdir(parasitized_dir)[0])
parasitized_image = imread(parasitized_cell)
plt.imshow(parasitized_image);

In [ ]:
# Calcular las dimensiones (alto y ancho) de todas las imágenes no infectadas
dim1 = []
dim2 = []
for image_filename in os.listdir(uninfected_dir):
    img = imread(os.path.join(uninfected_dir, image_filename))
    d1, d2, colors = img.shape
    dim1.append(d1)
    dim2.append(d2)

In [ ]:
# Graficar la distribución conjunta de anchos y altos para decidir el tamaño de redimensionamiento
sns.jointplot(x=dim1, y=dim2);

In [ ]:
# Mostrar las dimensiones promedio del dataset
print(np.mean(dim1))
print(np.mean(dim2))

In [ ]:
# Fijar el tamaño de redimensionamiento basado en la media de las dimensiones originales
image_width = 131
image_height = 131

In [ ]:
# Verificar el valor máximo de los píxeles en una imagen de ejemplo
# Nota: plt.imread() normaliza a float32 en rango [0, 1], pero los generadores de Keras
# (flow_from_directory) mantienen el rango nativo [0, 255].
parasitized_image.max()

### Data Augmentation

In [ ]:
# Configurar el generador de datos con aumentación para el conjunto de entrenamiento
datagen = ImageDataGenerator(
    rotation_range=45,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    zoom_range=0.1,
    fill_mode='nearest',
    horizontal_flip=True,
    vertical_flip=True
)

In [ ]:
# Generador de imágenes de entrenamiento con aumentación
train_generator = datagen.flow_from_directory(
    train_path,
    target_size=(image_height, image_width),
    color_mode='rgb',
    batch_size=32,
    class_mode='binary'
)

In [ ]:
# Generador de datos para test SIN aumentación: solo redimensiona las imágenes
test_datagen = ImageDataGenerator()

In [ ]:
# Generador de imágenes de prueba (sin shuffle para evaluación consistente)
test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(image_height, image_width),
    color_mode='rgb',
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

In [ ]:
# Verificar el mapeo de clases generado automáticamente por Keras
train_generator.class_indices

### Crear Modelo y Entrenamiento

In [ ]:
# Cargar EfficientNetB0 pre-entrenado en ImageNet SIN la cabeza clasificadora
# include_top=False remueve la capa densa final de 1000 clases de ImageNet
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',        # Usar pesos pre-entrenados (descarga automática la primera vez)
    input_shape=(224, 224, 3)  # Tamaño nativo de entrada que espera EfficientNetB0
)

# Congelar todas las capas del modelo base para entrenar solo la cabeza personalizada
base_model.trainable = False

In [ ]:
# Construir el modelo completo con transfer learning
# Las imágenes del dataset son 131x131; se redimensionan a 224x224 para EfficientNetB0
model = Sequential([
    Input(shape=(image_height, image_width, 3)),  # Entrada original del dataset: 131x131 RGB
    
    # Redimensionar imágenes de 131x131 a 224x224 para EfficientNetB0
    # interpolation='bilinear' es el método estándar de redimensionamiento
    Resizing(224, 224, interpolation='bilinear'),
    
    # Modelo base pre-entrenado (congelado en esta etapa)
    base_model,
    
    # Global Average Pooling: reduce cada mapa de características a un solo valor
    GlobalAveragePooling2D(),
    
    # Cabeza personalizada con regularización para clasificación binaria de células
    BatchNormalization(),
    Dropout(0.5),
    Dense(units=256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    # Capa de salida: 1 neurona con sigmoid (probabilidad para clasificación binaria)
    Dense(units=1, activation='sigmoid')
])

In [ ]:
# Compilar el modelo: binary_crossentropy es la loss estándar para clasificación binaria
# metrics=['accuracy'] permite monitorear la precisión durante el entrenamiento
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# EarlyStopping: detiene el entrenamiento si val_loss no mejora en 10 épocas consecutivas
# restore_best_weights=True recupera los pesos de la mejor época
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# Entrenar el modelo con los generadores y callbacks definidos
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=test_generator,
    validation_steps=len(test_generator),
    callbacks=[early_stop],
    epochs=50
)

### Evaluación

In [ ]:
# Convertir el historial de entrenamiento a DataFrame para graficar
history_df = pd.DataFrame(history.history)

In [ ]:
# Graficar la evolución de la accuracy en entrenamiento y validación
history_df[['accuracy', 'val_accuracy']].plot();

In [ ]:
# Graficar la evolución de la pérdida en entrenamiento y validación
history_df[['loss', 'val_loss']].plot();

In [ ]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(test_generator, steps=len(test_generator), verbose=0)
print(f'Loss en test: {loss:.4f}')
print(f'Accuracy en test: {accuracy:.4f}')

In [ ]:
# Generar predicciones sobre el conjunto de prueba y convertir probabilidades a clases (umbral 0.5)
predictions = (model.predict(test_generator, steps=len(test_generator)) > 0.5).astype('int32')

In [ ]:
# Reporte de clasificación: precision, recall y f1-score por clase
print(classification_report(test_generator.classes, predictions, target_names=['parasitized', 'uninfected']))

### Guardar Modelo

In [ ]:
# Guardar el modelo entrenado en formato nativo de Keras
model.save('malaria_model.keras')

### Cargar Modelo y Predecir Clases

In [ ]:
# Cargar el modelo guardado desde disco
malaria_model = load_model('malaria_model.keras')

In [ ]:
# Cargar una imagen individual y redimensionarla al tamaño esperado por el modelo
my_image = load_img(parasitized_cell, target_size=(image_height, image_width))
my_image = np.expand_dims(img_to_array(my_image), axis=0)
my_image.shape

In [ ]:
# Predecir la clase de la imagen cargada (0 = parasitized, 1 = uninfected)
prediction = (malaria_model.predict(my_image) > 0.5).astype('int32')
print(f'Predicción: {prediction[0][0]}')

In [ ]:
# Mostrar el diccionario de mapeo de clases para interpretar la predicción
train_generator.class_indices